<a href="https://colab.research.google.com/github/changsin/polysemy_xlang_wsi/blob/main/notebooks/polysemy_wsi_niv_cuv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Measuring Polysemy Across Languages
Polysemy, the capacity of a word to have more than one sense, is fundamental to natural language. This project uses WSI (Word Sense Induction) to compare and analyze polysemy across languages, starting from English and Chinese.

** NB **
Before running, create **bible_data** folder and upload the following files:
- **english_verses.txt**
- **chinese_verses.txt**
- **jieba_biblical_dict.txt**: for the Chinese corpus.
- **kjv_archaic_exclude.txt**: for KJV.

A verses file contains Bible verses with their book, chapter, and verse number in CSV format like the following.
```
verse_id,text
Gen.1.1,In the beginning God created the heavens and the earth.
```


## Steps
**Step 0: Install dependencies**

**Step 1: Corpus Preparation** (done outside of the notebook) Parses plain-text Bible files to output `language_verses.txt` file with the format: verse_id, text

**Step 2: Preprocessing - Extract Common Nouns** Process English and Chinese corpora to extract common nouns using spaCy (English) and jieba (Chinese) to tokenize, POS-tag, and lemmatize words.

**Step 3: Extract Context Embeddings** Generate contextual embeddings for each noun occurence using a pre-trained multilingual transformer model (XML-RoBERTa-base).

**Step 4: WSI Clustering** Apply HDBSCAN clustering to the extracted contextual embeddings and identify word senses based on the identified clusters.

**Step 5: Validation and Statistical Analysis** Validate the induced senses against WordNet and run statistical comparison of the two languages (e.g., Mann-Whitney U test)

**Step 6: Qualitative Analysis** Create a qualitative report of the most polysemous words, example contexts for each induced sense.


# Step 0. Install dependencies

In [ ]:
!pip install torch transformers scikit-learn numpy pandas --break-system-packages

In [ ]:
!pip install scipy matplotlib seaborn

In [ ]:
!pip install nltk spacy jieba hdbscan

In [ ]:
!python -m spacy download en_core_web_sm
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

# Step 2. Preprocessing: Extract Common Nouns

In [ ]:
import json
import re
import pandas as pd
from pathlib import Path
from collections import Counter

# ─── Configuration ────────────────────────────────────────────────────────────

DATA_DIR = Path("/content") / "bible_data"
MIN_FREQ = 30          # Minimum occurrences per lemma for WSI
MAX_CONTEXT_LEN = 512  # Characters — prevents overlong inputs to transformers


# ─── Frequency Filtering ──────────────────────────────────────────────────────


def apply_frequency_filter(df: pd.DataFrame, min_freq: int = MIN_FREQ) -> tuple:
    """
    Keep only lemmas appearing at least `min_freq` times.
    Returns (filtered_df, freq_df).
    """
    freq = df.groupby("lemma").size().reset_index(name="count")
    freq = freq.sort_values("count", ascending=False)
    valid_lemmas = set(freq[freq["count"] >= min_freq]["lemma"])
    filtered = df[df["lemma"].isin(valid_lemmas)].copy()
    return filtered, freq

In [ ]:
KJV_ARCHAIC_EXCLUDE_FILE = DATA_DIR / "kjv_archaic_exclude.json"
# ZH_STOPWORDS_FILE = DATA_DIR / "chinese_stopwords.json"

EN_THEOLOGICAL_EXCLUDE = {
    # Lemma-level backstop for cases where spaCy tags as NOUN not PROPN
    "spirit",    # "Holy Spirit" — spaCy inconsistently tags as NOUN
    "ghost",     # "Holy Ghost" (KJV form; rare in NIV but present)
}

def _load_set_from_json(file_path: Path) -> set:
    """Loads a set of strings from a JSON file."""
    if not file_path.exists():
        print(f"WARNING: Exclusion file not found at {file_path}. Returning empty set.", flush=True)
        return set()
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise TypeError(f"Expected JSON file at {file_path} to contain a list, but got {type(data)}")
    return set(data)

# KJV-specific: exclude archaic pronouns/verbs mis-tagged as nouns
KJV_ARCHAIC_EXCLUDE = _load_set_from_json(KJV_ARCHAIC_EXCLUDE_FILE)


# ─── English Preprocessing ────────────────────────────────────────────────────

def preprocess_english(verse_csv: Path) -> pd.DataFrame:
    """
    Process English verses with spaCy.
    Returns long-format DataFrame: one row per noun occurrence.
    """
    try:
        import spacy
    except ImportError:
        raise ImportError("Run: pip install spacy --break-system-packages && python -m spacy download en_core_web_sm")

    print("  [EN] Loading spaCy model…")
    nlp = spacy.load("en_core_web_sm", disable=["ner", "textcat"])

    df = pd.read_csv(verse_csv)
    print(f"  [EN] Processing {len(df):,} verses…")

    import time
    records   = []
    texts     = df["text"].tolist()
    verse_ids = df["verse_id"].tolist()
    total     = len(texts)
    t0        = time.time()

    for i, (doc, vid) in enumerate(zip(nlp.pipe(texts, batch_size=512), verse_ids), 1):
        context = doc.text[:MAX_CONTEXT_LEN]
        for token in doc:
            # Keep only common nouns; exclude proper nouns and pronouns
            if (
                token.pos_ == "NOUN"
                and not token.is_stop
                and not token.is_punct
                and len(token.lemma_) > 1
                and token.lemma_.isalpha()
                and token.lemma_.lower() not in EN_THEOLOGICAL_EXCLUDE
                ## for KJV processing-only
                and token.lemma_.lower() not in KJV_ARCHAIC_EXCLUDE
            ):
                records.append({
                    "verse_id": vid,
                    "token":    token.text,
                    "lemma":    token.lemma_.lower(),
                    "context":  context,
                })

        # report the progress
        if i % 1000 == 0 or i == total:
            elapsed = time.time() - t0
            rate    = i / elapsed if elapsed > 0 else 0
            eta_min = (total - i) / rate / 60 if rate > 0 else 0
            print(f"    … {i:,}/{total:,} verses  "
                  f"({rate:.0f} v/s)  "
                  f"ETA {eta_min:.1f} min  "
                  f"nouns so far: {len(records):,}",
                  end="\r")

    elapsed_total = time.time() - t0
    print(f"\n  [EN] Done in {elapsed_total/60:.1f} min. "
          f"Extracted {len(records):,} noun occurrences.")
    return pd.DataFrame(records)

In [ ]:
# Chinese POS tags for common nouns (jieba.posseg notation)
ZH_NOUN_PREFIXES = {"n"}           # Common noun prefix
ZH_EXCLUDE_TAGS  = {"nr", "ns", "nt", "nz", "nw", "nrt"}  # Proper nouns to exclude

# 标签  含义      标签	  含义		标签	含义	  	标签	含义
# n		普通名词   f	   方位名词	  s		 处所名词	 t	    时间
# nr	人名	    ns	  地名	    nt	   机构名	   nw	  作品名
# nz	其他专名   v     普通动词	  vd     动副词		vn	   名动词
# a		形容词     ad	   副形词	 	 an		名形词	    d	   副词
# m		数量词     q		 量词	    r	   代词		  p		 介词
# c		连词	     u		 助词		xc	  其他虚词     w	 标点符号
# PER	人名       LOC	 地名	    ORG	   机构名	   TIME   时间

# ── Theological proper noun exclusion lists ───────────────────────────────────
# These terms are proper nouns in English (God, Lord, Christ etc.) — excluded
# by spaCy's PROPN tag — but are tagged as common nouns n by jieba in Chinese
# due to the absence of capitalisation. They must be excluded explicitly from
# the Chinese data to ensure cross-lingual comparability.
#
# English side: spaCy correctly tags God/Lord/Christ as PROPN (excluded).
# Exception: "Spirit" (Holy Spirit) is sometimes tagged NOUN by spaCy, so it
# is added to EN_THEOLOGICAL_EXCLUDE as a lemma-level backstop.
#
# Borderline cases kept in both languages:
#   先知/prophet  — generic occupational noun, polysemous, common in both
#   天使/angel    — generic supernatural being, common noun in both
#   魔鬼/devil    — common noun in EN; jieba tags n in ZH

ZH_THEOLOGICAL_EXCLUDE = {
    # Core deity names / titles
    "神",     # God (most frequent — 1244 occurrences)
    "主",     # Lord
    "上帝",   # God (formal)
    "耶和華", # Yahweh / LORD
    "基督",   # Christ
    "耶穌",   # Jesus (also usually tagged nr, but belt-and-suspenders)
    "聖靈",   # Holy Spirit
    "聖神",   # Holy Spirit (alternate form in some CUV editions)
    "彌賽亞", # Messiah
    # Adversarial proper nouns
    "撒但",   # Satan
    "別西卜", # Beelzebub
    # Exception for mis-tagged verbal phrases that Jieba assigns 'n'
    "主說",   # 'The Lord says' - incorrectly tagged as a noun by Jieba
    "亞瑪力", # Amalek or Amalekite - Proper noun, often mis-tagged as 'n'

    # ═══════════════════════════════════════════════════════════
    # Compositional patterns that jieba won't force-split
    # ═══════════════════════════════════════════════════════════

    # Subject + nominalizer 所 (compositional relative clauses)
    "神所",   # God + what (k=3, n=132: frame-splitting by verb)
    "主所",   # Lord + what (if exists)
    "人所",   # people + what (if exists)
    # Note: 聖所 "sanctuary" and 居所 "dwelling" are lexicalized - NOT excluded

    # Nominalizer 所 + verb (classical relative clauses)
    "所作",   # what is done (k=2, n=52)
    "所獻",   # what is offered (k=1, n=36)
    "所求",   # what is sought (if exists)
    "所犯",

    # Verb + object patterns
    "生子",   # give birth to son (k=1, n=32)
    "生兒",   # give birth to child (if exists)
    "定睛",   # fix eyes (verb phrase)
    "行事",   # conduct affairs

    "經上", "人陷", "對面",

    # Proper noun fragments (jieba fragments despite 100000 frequency)
    "基利",   # Fragment of 基利心山
    "大利",   # Fragment of 大利烏, 大利拉
    "米利",   # Fragment of 米利暗
    "米拉",   # Fragment of 米拉利
    "比利",   # Fragment of 比利提人, 比利人
    "示巴",   # Fragment of 拔示巴

    # Other compositional patterns
    "二人",   # two people (numeral + noun)
    "別神",   # other gods (modifier + noun)
    "木作",   # woodwork (verb-object or specialized noun)
}

# Path to custom jieba dictionary for biblical proper names
JIEBA_DICT_PATH = DATA_DIR / "jieba_biblical_dict.txt"

# ── Post-segmentation POS correction ─────────────────────────────────────────
# jieba's POS tagger runs independently of the segmentation dictionary and
# can assign incorrect tags even for dictionary entries. These words are
# forced to tag n after segmentation regardless of what the POS tagger assigned.
#
# 地: jieba assigns uv (虛詞/copular particle) in classical subject-predicate
#     constructions like 地是空虛混沌 (Gen.1.2) because it parses 地 as a
#     topic marker rather than a subject noun. This is a known jieba limitation
#     with literary Chinese. Since English "earth" (freq=739) is always tagged
#     NOUN by spaCy, forcing 地 to n is required for cross-lingual comparability.
ZH_FORCE_NOUN_TAG = {
    "地",   # earth/ground/land — incorrectly tagged uv in copular constructions
}

# ─── Chinese Preprocessing ────────────────────────────────────────────────────


def preprocess_chinese(verse_csv: Path) -> pd.DataFrame:
    """
    Process Chinese verses with jieba (word segmentation + POS tagging).
    Returns long-format DataFrame: one row per noun occurrence.

    Three-part preprocessing strategy:
    1. Custom dictionary - fixes segmentation and marks correct POS
    2. Forced splits - breaks apart compositional phrases
    3. POS filtering - only keeps common nouns (tag 'n')
    """
    import time
    try:
        import jieba
        import jieba.posseg as pseg
    except ImportError:
        raise ImportError("Run: pip install jieba --break-system-packages")

    # Silence jieba logging FIRST
    jieba.setLogLevel("ERROR")

    # Load custom dictionary
    if JIEBA_DICT_PATH.exists():
        jieba.load_userdict(str(JIEBA_DICT_PATH))
        print(f"  [ZH] Loaded custom dictionary: {JIEBA_DICT_PATH.name}", flush=True)
    else:
        print(f"  [ZH] WARNING: custom dictionary not found at {JIEBA_DICT_PATH}", flush=True)

    # ================================================================
    # FORCE SPLITS - Compositional phrases that should be separate tokens
    # ================================================================

    # Subject + Verb patterns
    subject_verb = [
        ('人', '陷'),     # people + fall
        ('人', '行'),     # people + walk/do
        ('生', '兒'),     # give birth + child
        ('工', '都'),     # work + all
    ]

    # Modifier + Noun
    modifier_noun = [
        ('全', '會眾'),   # entire + congregation
        ('眾', '教會'),   # many + churches
    ]

    # Preposition/Locative + Noun
    locative = [
        ('在', '世'),     # at + world
        ('經', '上'),     # scripture + on
        ('床', '上'),     # bed + on
        ('對', '面'),     # opposite + face
        ('面', '向'),     # face + toward
    ]

    # Possessive constructions
    possessive_splits = [
        ('我', '心'),     # my + heart
    ]

    # Negation + Verb
    negation_verb = [
        ('不', '容'),     # not + permit
    ]

    # Combine all splits
    all_splits = (subject_verb + modifier_noun + locative
                  + possessive_splits + negation_verb)

    print(f"  [ZH] Applying {len(all_splits)} forced segmentation splits...", flush=True)
    for word_tuple in all_splits:
        jieba.suggest_freq(word_tuple, True)

    # Initialize jieba
    print("  [ZH] Building jieba model (one-time, ~10-30s)...", flush=True)
    jieba.initialize()
    print("  [ZH] Model ready.", flush=True)

    zh_stopwords = _load_chinese_stopwords()

    df = pd.read_csv(verse_csv)
    total = len(df)
    print(f"  [ZH] Processing {total:,} verses...", flush=True)
    t0 = time.time()

    records = []
    for i, row in enumerate(df.itertuples(index=False), 1):
        verse_id = row.verse_id
        text = str(row.text)
        context = text[:MAX_CONTEXT_LEN]

        for word, flag in pseg.cut(text):
            flag_str = str(flag)

            # Force known mis-tagged tokens to correct POS
            if word in ZH_FORCE_NOUN_TAG:
                flag_str = "n"

            if (
                flag_str[:1] in ZH_NOUN_PREFIXES
                and flag_str not in ZH_EXCLUDE_TAGS
                and word not in zh_stopwords
                and word not in ZH_THEOLOGICAL_EXCLUDE
                and len(word) >= 1
                and not word.isdigit()
            ):
                records.append({
                    "verse_id": verse_id,
                    "token": word,
                    "lemma": word,
                    "context": context,
                })

        # Progress reporting
        if i % 100 == 0 or i == total:
            elapsed = time.time() - t0
            rate = i / elapsed if elapsed > 0 else 0
            eta_min = (total - i) / rate / 60 if rate > 0 else 0
            print(
                f"    ... {i:,}/{total:,} verses"
                f"  ({rate:.0f} v/s)"
                f"  ETA {eta_min:.1f} min"
                f"  nouns: {len(records):,}",
                flush=True
            )

    elapsed_total = time.time() - t0
    print(f"  [ZH] Done in {elapsed_total/60:.1f} min. "
          f"Extracted {len(records):,} noun occurrences.")
    return pd.DataFrame(records)

def _load_chinese_stopwords() -> set:
    """
    Chinese function word stoplist for CUV Traditional (CHT) text.

    Design decisions:
    ─────────────────────────────────────────────────────────────
    1. CHT variants included alongside CHS equivalents for all
       characters that differ between scripts (說/说, 會/会, etc.)

    2. 人 is NOT a stopword. It is a genuine common noun meaning
       "person / people / man / humanity" and is highly polysemous
       in biblical text. Jieba tags it as n (common noun) in most
       contexts, so it passes the POS filter correctly. Removing it
       would discard one of the most semantically rich words in the
       corpus.

    3. Pronouns (他/她/祂/你/我 etc.) are NOT listed here. They are
       tagged by jieba as r (pronoun), which is already excluded by
       the POS filter (we keep only n* tags). Listing them would be
       redundant. The various gendered and honorific variants
       (他/她/它/祂) all carry the r tag and are excluded uniformly.

    4. This list covers only high-frequency grammatical function
       words that jieba may occasionally mis-tag as nouns.
       It is intentionally conservative.
    ─────────────────────────────────────────────────────────────
    """
    return {
        # Structural particles (occasionally mis-tagged as n by jieba)
        # NOTE: 地 is intentionally NOT listed here.
        # In CUV literary style, 地 is overwhelmingly used as a noun
        # (earth/land/ground) matching English "earth" (freq=739).
        # The adverbial particle use of 地 is rare in classical biblical text.
        # Removing it would create an asymmetry with English where "earth"
        # is correctly retained as a high-frequency common noun.
        "的", "得",
        # Aspect markers — CHT: 著, CHS: 着
        "了", "著", "着",
        # Conjunctions / connectives
        "和", "與", "与", "及", "或", "但", "而", "且",
        # Adverbs sometimes mis-tagged
        "也", "都", "就", "才", "又", "還", "还", "已",
        "很", "更", "最", "太", "非常",
        # Adjectives sometimes mis-tagged
        "大", "小",
        # Negation
        "不", "沒有", "没有", "未", "無", "无",
        # Existential / copular
        "是", "有", "在",
        # Determiners / quantifiers
        "一", "這", "这", "那", "各", "每", "某", "其",
        # Directional / locative words with no sense variation
        "上", "下", "中", "內", "内", "外", "前", "後", "后",
        "裡", "里", "間", "间",
        # Common verbs jieba occasionally tags as nouns in CUV
        "說", "说", "看", "去", "來", "来", "到", "給", "给",
        "要", "會", "会",

        "全",   # all, whole (determiner)
        "阿",   # prefix particle
        "為",   # to

        # Common verbs jieba occasionally tags as nouns
        "行",     # walk, do, conduct
        "作",     # do, make

        # Temporal/ordinal markers
        "先",     # first, before
        "甲",     # first (天干 ordinal)
        "亞",     # second (ordinal), also prefix in 亞洲, 亞伯拉罕 etc.
    }

In [ ]:
import jieba
import jieba.posseg as pseg

jieba.load_userdict("bible_data/jieba_biblical_dict.txt")

# Test known proper nouns
test_cases = [
    "巴勒對以色列",
    "拜巴力的人",
    "神說",
    "先知說",
    "主說",
    "所作所為",
    "挪亞就這樣行．凡 神所吩咐的、他都照樣行了。"
]

for text in test_cases:
    print(f"\n{text}")
    for word, flag in pseg.cut(text):
        print(f"  {word}/{flag}")
# ```

# **Expected:**
# ```
# 巴勒對以色列
#   巴勒/nr   ← Should be nr (proper noun)

# 拜巴力的人
#   拜/v
#   巴力/nr   ← Should be nr (proper noun)
# ```

# **If you're getting:**
# ```
#   巴勒/n    ← WRONG
#   巴力/n    ← WRONG

In [ ]:
print("=" * 60)
print("Step 2: Preprocessing")
print("=" * 60)

# ── English ───────────────────────────────────────────────────
en_raw = preprocess_english(DATA_DIR / "english_verses.csv")
en_filtered, en_freq = apply_frequency_filter(en_raw)
en_filtered.to_csv(DATA_DIR / "english_nouns.csv", index=False, encoding="utf-8")
en_freq.to_csv(DATA_DIR / "english_noun_freq.csv", index=False, encoding="utf-8")
print(f"  [EN] {en_filtered['lemma'].nunique():,} lemmas ≥ {MIN_FREQ} occurrences retained.")

# ── Summary ───────────────────────────────────────────────────
print("\n── Preprocessing Summary ──")
print(f"  EN noun tokens (filtered) : {len(en_filtered):,}")
print(f"  EN unique lemmas          : {en_filtered['lemma'].nunique():,}")

print("\n  Top 10 English nouns:")
print(en_freq.head(10).to_string(index=False))

print("\n✓ Step 2 complete.\n")

Step 2: Preprocessing
  [EN] Loading spaCy model…
  [EN] Processing 31,088 verses…
    … 31,088/31,088 verses  (442 v/s)  ETA 0.0 min  nouns so far: 116,878
  [EN] Done in 1.2 min. Extracted 116,878 noun occurrences.
  [EN] 636 lemmas ≥ 30 occurrences retained.

── Preprocessing Summary ──
  EN noun tokens (filtered) : 98,195
  EN unique lemmas          : 636

  Top 10 English nouns:
   lemma  count
     man   3927
     son   2929
    king   2533
  people   2376
     day   2037
    land   1501
    hand   1296
  father   1289
   house   1054
offering   1026

✓ Step 2 complete.



In [ ]:
print("=" * 60)
print("Step 2: Preprocessing")
print("=" * 60)

# ── Chinese ───────────────────────────────────────────────────
zh_raw = preprocess_chinese(DATA_DIR / "chinese_verses.csv")
zh_filtered, zh_freq = apply_frequency_filter(zh_raw)
zh_filtered.to_csv(DATA_DIR / "chinese_nouns.csv", index=False, encoding="utf-8")
zh_freq.to_csv(DATA_DIR / "chinese_noun_freq.csv", index=False, encoding="utf-8")
print(f"  [ZH] {zh_filtered['lemma'].nunique():,} lemmas ≥ {MIN_FREQ} occurrences retained.")

# ── Summary ───────────────────────────────────────────────────
print("\n── Preprocessing Summary ──")
print(f"  ZH noun tokens (filtered) : {len(zh_filtered):,}")
print(f"  ZH unique lemmas          : {zh_filtered['lemma'].nunique():,}")

print("\n  Top 10 Chinese nouns:")
print(zh_freq.head(10).to_string(index=False))

print("\n✓ Step 2 complete.\n")

# Step 3: Extract Context Embeddings

## 3.1 get_target_embedding()
The core function is `get_target_embedding().`
It extracts contextualized embeddings for specific target words within sentences using a pre-trained transformer model (like `xlm-roberta-base` in this notebook). This is a crucial step for Word Sense Induction (WSI) as it captures the meaning of a word based on its surrounding context.

Here's a step-by-step explanation:

1.  **Purpose**: The primary goal is to generate a numerical vector (embedding) for each occurrence of a target word in a sentence. This vector represents the word's meaning in that particular context.

2.  **Inputs**: It takes a list of `sentences` and a corresponding list of `target_words`. The `tokenizer` and `model` (pre-loaded `AutoTokenizer` and `AutoModel` from `transformers`) are also passed as arguments.

3.  **Batch Processing**: To efficiently process a large number of sentences, the function uses mini-batches (defined by `BATCH_SIZE`). This allows for parallel computation on the GPU (if available).

4.  **Tokenization**: For each batch, sentences are tokenized using the `tokenizer`. This converts raw text into numerical `input_ids` that the model understands. Crucially, `return_offsets_mapping=True` is used to map the subword tokens back to their original character positions in the text, which helps in identifying the target word's tokens later.

5.  **Model Inference**: The tokenized inputs are fed into the `model` to get its hidden states. The `outputs.hidden_states` contain the activations from all layers of the transformer model.

6.  **Embedding Extraction from Layers**: Standard WSI practice often involves averaging embeddings from the last few layers of a transformer. In this function, the `LAYERS` variable (e.g., `[-1, -2, -3, -4]`) specifies which layers to consider. The hidden states from these selected layers are stacked and then averaged (`.mean(dim=0)`) to get a single contextual representation for each token in the sentence.

7.  **Target Word Subword Alignment**: The helper function `_find_subword_positions()` is used to locate the subword tokens that correspond to the `target_word` within the tokenized sentence. This is important because a single word might be split into multiple subword tokens by the tokenizer (e.g., "embedding" -> "em", "##bedd", "##ing").

8.  **Mean-Pooling for Target Word**: If the `target_word` is found and consists of multiple subword tokens, their corresponding contextual embeddings (from the averaged layers) are mean-pooled to create a single embedding for the target word. If the target word isn't found (e.g., due to tokenization quirks), it falls back to mean-pooling the entire sentence's non-special tokens.

9.  **L2 Normalization**: Finally, all extracted embeddings are L2 normalized. This is a common practice in WSI and other NLP tasks, especially when using cosine similarity for distance calculations, as it ensures that the length of the vector does not influence the similarity.

10. **Output**: The function returns a NumPy array (`np.ndarray`) where each row is the L2-normalized contextual embedding for a target word occurrence.

## 3.2 model inference output is embeddings

Context embeddings are extracted when the model inferences: i.e., `outputs = model(**encoded)`. It is important to note the role `torch.no_grad()` context manager plays when inferencing. Here's what it does:

1.  **Disables Gradient Calculation**: When you're training a neural network, PyTorch's autograd engine automatically tracks all operations on tensors that have `requires_grad=True`. This is necessary to compute gradients during the backward pass for optimization (e.g., using Stochastic Gradient Descent).

2.  **Why Disable it During Inference?**
    *   **Memory Efficiency**: Calculating and storing gradients requires significant memory. During inference, we don't need to update model weights, so these gradients are unnecessary. Disabling gradient calculation frees up GPU memory, allowing for larger batch sizes or processing larger models.
    *   **Speed**: The autograd engine adds computational overhead to track operations. By disabling it, computations can run faster as PyTorch doesn't need to build the computation graph.
    *   **Preventing Accidental Updates**: It ensures that no gradients are computed and, consequently, no model parameters are accidentally updated, which is crucial for maintaining a trained model's state during evaluation.

In summary, `with torch.no_grad():` tells PyTorch, "Hey, for everything inside this block, I don't need to compute gradients, so save memory and compute faster!" This is standard practice for evaluation, prediction, and embedding extraction tasks like the one in this notebook.

## 3.3 _find_subword_positions()
The _find_subword_positions() function serves a very specific and important purpose in the embedding extraction process. It acts as a utility to correctly identify the location of the target word's subword tokens within the full tokenized sentence.

Here's a breakdown:

Input: It takes two lists of integers:

sentence_ids: The list of token IDs for the entire sentence (including special tokens like [CLS] and [SEP]).
target_ids: The list of token IDs that represent the target word when tokenized in isolation.
The Problem it Solves: Transformer tokenizers (like the one used for XLM-RoBERTa) often break down words into smaller subword units (e.g., "embedding" might become "em", "##bedd", "##ing"). When we have a target word, we need to find exactly which of the sentence's subword tokens correspond to that target word, especially if the target word itself is a multi-subword unit.

How it Works: The function searches for target_ids as a contiguous subsequence within sentence_ids. It iterates through sentence_ids and checks if the target_ids sequence matches a slice of sentence_ids of the same length.

Output: If a match is found, it returns a list of integer indices representing the starting and ending positions of the target word's subword tokens within the sentence_ids. If no match is found (which can happen due to various tokenization edge cases or if the target word wasn't fully contained in the processed sentence length), it returns an empty list [].

Why it's important: Once these positions are identified, the get_target_embedding() function can then select the corresponding hidden states (contextual embeddings) from the model's output for only those specific subword tokens and mean-pool them to get a single, accurate contextual embedding for the entire target word.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import time # Import time module for progress tracking
from pathlib import Path
from transformers import AutoTokenizer, AutoModel
from typing import List, Tuple

# ─── Configuration ────────────────────────────────────────────────────────────

DATA_DIR    = Path("/content") / "bible_data"
MODEL_NAME  = "xlm-roberta-base"   # Multilingual; same model for EN and ZH
BATCH_SIZE  = 32                   # Reduce to 8-16 if OOM on CPU
LAYERS      = [-1, -2, -3, -4]     # Last 4 layers averaged (standard WSI practice)
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
MAX_SEQ_LEN = 128                  # Max subword tokens per sentence

print(f"Using device: {DEVICE}")

# ─── Model Loading ────────────────────────────────────────────────────────────

def load_model():
    print(f"  [model] Loading {MODEL_NAME}…")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model     = AutoModel.from_pretrained(MODEL_NAME, output_hidden_states=True)
    model.to(DEVICE)
    model.eval()
    return tokenizer, model


# ─── Embedding Extraction ─────────────────────────────────────────────────────

def get_target_embedding(
    tokenizer,
    model,
    sentences:  List[str],
    target_words: List[str],
) -> np.ndarray:
    """
    For each (sentence, target_word) pair, extract the contextual embedding
    of the target by:
      1. Tokenizing the sentence
      2. Finding subword token positions for the target word
      3. Averaging hidden states across the last 4 layers at those positions
      4. Mean-pooling across subwords for multi-token targets

    Returns: np.ndarray of shape (N, hidden_dim)
    """
    print(f"  Starting embedding extraction for {len(sentences):,} sentences...", flush=True)
    all_embeddings = []
    total_sentences = len(sentences)
    t0 = time.time()

    for i in range(0, total_sentences, BATCH_SIZE):
        batch_sents  = sentences[i : i + BATCH_SIZE]
        batch_targets = target_words[i : i + BATCH_SIZE]

        encoded = tokenizer(
            batch_sents,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LEN,
            return_offsets_mapping=True,
        )
        offset_mappings = encoded.pop("offset_mapping")  # not passed to model

        encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

        with torch.no_grad():
            outputs = model(**encoded)

        # Stack selected hidden layers: shape (n_layers, batch, seq_len, hidden)
        hidden_states = torch.stack(
            [outputs.hidden_states[l] for l in LAYERS], dim=0
        )
        # Mean over selected layers: (batch, seq_len, hidden)
        layer_mean = hidden_states.mean(dim=0).cpu().numpy()

        input_ids = encoded["input_ids"].cpu().numpy()

        for j, (target, offsets_j) in enumerate(zip(batch_targets, offset_mappings)):
            # Re-encode the target word alone to find its subword tokens
            target_enc = tokenizer.encode(
                target, add_special_tokens=False
            )
            # Find target subword positions in the sentence encoding
            target_positions = _find_subword_positions(
                input_ids[j].tolist(), target_enc
            )
            if target_positions:
                token_emb = layer_mean[j][target_positions].mean(axis=0)
            else:
                # Fallback: mean-pool entire sequence (excluding [CLS]/[SEP])
                seq_len = (input_ids[j] != tokenizer.pad_token_id).sum()
                token_emb = layer_mean[j][1 : seq_len - 1].mean(axis=0)

            all_embeddings.append(token_emb)

        # Progress indicator (print every 1000 sentences or at the end)
        current_processed = i + len(batch_sents)
        if current_processed % 1000 == 0 or current_processed == total_sentences:
            elapsed_time = time.time() - t0
            rate = current_processed / elapsed_time if elapsed_time > 0 else 0
            eta_seconds = (total_sentences - current_processed) / rate if rate > 0 else 0
            eta_minutes = eta_seconds / 60
            print(f"    ... {current_processed:,}/{total_sentences:,} sentences "
                  f"({rate:.0f} s/s) "
                  f"ETA {eta_minutes:.1f} min ", flush=True)

    print(f"  Finished embedding extraction. Total {len(all_embeddings):,} embeddings.", flush=True)
    embeddings = np.array(all_embeddings, dtype=np.float32)
    # L2 normalize for cosine-based clustering
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1, norms)
    return embeddings / norms


def _find_subword_positions(
    sentence_ids: List[int], target_ids: List[int]
) -> List[int]:
    """Find the start position of `target_ids` as a subsequence in `sentence_ids`."""
    n, m = len(sentence_ids), len(target_ids)
    for start in range(n - m + 1):
        if sentence_ids[start : start + m] == target_ids:
            return list(range(start, start + m))
    return []


# ─── Per-language Pipeline ────────────────────────────────────────────────────

def extract_embeddings_for_language(
    lang: str,
    noun_csv: Path,
    tokenizer,
    model,
) -> None:
    """Load nouns, extract embeddings, and save as .npz."""
    out_path = DATA_DIR / f"{lang}_embeddings.npz"
    if out_path.exists():
        print(f"  [{lang.upper()}] Embeddings already exist — skipping.")
        return

    df = pd.read_csv(noun_csv)
    print(f"  [{lang.upper()}] Extracting embeddings for {len(df):,} noun occurrences…")

    embeddings = get_target_embedding(
        tokenizer,
        model,
        sentences    = df["context"].tolist(),
        target_words = df["token"].tolist(),
    )
    print()  # newline after progress indicator

    np.savez_compressed(
        out_path,
        embeddings = embeddings,
        lemmas     = df["lemma"].to_numpy(dtype=str),
        verse_ids  = df["verse_id"].to_numpy(dtype=str),
        tokens     = df["token"].to_numpy(dtype=str),
    )
    print(f"  [{lang.upper()}] Saved {embeddings.shape} embeddings → {out_path.name}")

In [ ]:
print("=" * 60)
print("Step 3: Contextual Embedding Extraction (XLM-R)")
print("=" * 60)

tokenizer, model = load_model()

extract_embeddings_for_language(
    "english",
    DATA_DIR / "english_nouns.csv",
    tokenizer, model,
)

print("\n✓ Step 3 complete.\n")

In [ ]:
print("=" * 60)
print("Step 3: Contextual Embedding Extraction (XLM-R)")
print("=" * 60)

tokenizer, model = load_model()

extract_embeddings_for_language(
    "chinese",
    DATA_DIR / "chinese_nouns.csv",
    tokenizer, model,
)

print("\n✓ Step 3 complete.\n")

# Step 4: WSI Clustering

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize
from typing import Tuple, Dict
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# ─── Configuration ────────────────────────────────────────────────────────────

DATA_DIR      = Path("/content") / "bible_data"

RANDOM_STATE  = 42
# Reduce using PCA if too many samples
LARGE_SAMPLE_SIZE = 200
# Min number of samples to form a cluster
MIN_SAMPLE_SIZE = 6
# Min percentage of samples to be a cluster
MIN_SAMPLE_PERCENT_FOR_CLUSTER = 0.03
# If the second largest cluster is below this proportion, count as monosemous
SECOND_CLUSTER_THRESHOLD = 0.08
NOISE_LABEL = -1

# ─── HDBSCAN ──────────────────────────────────────────────

import hdbscan
from sklearn.decomposition import PCA

def induce_senses_hdbscan(embeddings: np.ndarray):
    n_samples, n_features = embeddings.shape

    if n_samples < 20:
        return 1, np.zeros(n_samples, dtype=int)

    # L2 normalize
    embeddings = normalize(embeddings)

    # Optional PCA only if large sample size
    if n_samples > LARGE_SAMPLE_SIZE:
        n_components = min(100, n_samples - 1, n_features)
        embeddings = PCA(
            n_components=n_components,
            random_state=RANDOM_STATE
        ).fit_transform(embeddings)
        embeddings = normalize(embeddings)

    # too conservative
    # min_cluster_size = max(8, int(0.05 * n_samples))
    # 3% instead of 5% (too conservative)
    min_cluster_size = max(MIN_SAMPLE_SIZE,
                           int(MIN_SAMPLE_PERCENT_FOR_CLUSTER * n_samples))

    clusterer = hdbscan.HDBSCAN(
        metric="euclidean",
        min_cluster_size=min_cluster_size,
        min_samples=max(5, min_cluster_size // 2),
        cluster_selection_method="eom"
    )

    labels = clusterer.fit_predict(embeddings)

    clusters = set(labels)
    # discard if noise
    clusters.discard(NOISE_LABEL)

    k = len(clusters)

    if k <= 1:
        return 1, np.zeros(n_samples, dtype=int)

    sizes = sorted([sum(labels == c) for c in clusters], reverse=True)

    # Use the configurable variable
    if sizes[1] / n_samples < SECOND_CLUSTER_THRESHOLD:
        return 1, np.zeros(n_samples, dtype=int)

    return k, labels


# ─── Run WSI for entire language ──────────────────────────────────────────────

def run_wsi_for_language(lang: str):
    """
    L Load embeddings and run WSI for all lemmas in a language.
    Saves two files:
      1. {lang}_wsi_results.csv    - summary stats per lemma
      2. {lang}_sense_labels.csv   - cluster assignments per occurrence
    """
    embeddings_file = DATA_DIR / f"{lang}_embeddings.npz"
    nouns_file      = DATA_DIR / f"{lang}_nouns.csv"

    if not embeddings_file.exists():
        print(f"  [{lang.upper()}] ERROR: {embeddings_file.name} not found. Run Step 3 first.")
        return

    print(f"\n  [{lang.upper()}] Loading embeddings…")
    data = np.load(embeddings_file, allow_pickle=True)
    lemmas_array = data["lemmas"]
    embeddings   = data["embeddings"]
    verse_ids    = data["verse_ids"]
    unique_lemmas = np.unique(lemmas_array)

    print(f"  [{lang.upper()}] Running WSI for {len(unique_lemmas):,} lemmas…")

    results = []
    all_labels = []  # Collect per-instance labels for sense_labels.csv

    for i, lemma in enumerate(unique_lemmas):
        mask = (lemmas_array == lemma)
        lemma_embeds = embeddings[mask]
        lemma_vids   = verse_ids[mask]

        k_hdb, labels_hdb = induce_senses_hdbscan(lemma_embeds)

        results.append({
            "lemma": lemma,
            "n_instances": len(lemma_embeds),
            "k_hdbscan": k_hdb,
        })

        for vid, cluster in zip(lemma_vids, labels_hdb):
            all_labels.append({
                "lemma": lemma,
                "verse_id": vid,
                "cluster_hdbscan": int(cluster),
            })

        if (i + 1) % 50 == 0 or (i + 1) == len(unique_lemmas):
            print(f"    … {i + 1:,}/{len(unique_lemmas):,} lemmas", end="\r")


    # Save summary results
    df = pd.DataFrame(results)
    out_path = DATA_DIR / f"{lang}_wsi_results.csv"
    df.to_csv(out_path, index=False)
    print(f"\n  [{lang.upper()}] Saved: {out_path.name}")

    # Save per-instance cluster labels
    labels_df = pd.DataFrame(all_labels)
    labels_path = DATA_DIR / f"{lang}_sense_labels.csv"
    labels_df.to_csv(labels_path, index=False)
    print(f"  [{lang.upper()}] Saved: {labels_path.name}")

    print(f"  [{lang.upper()}] Mean k (HDBSCAN): {df['k_hdbscan'].mean():.2f}  "
      f"Median: {df['k_hdbscan'].median():.0f}  "
      f"Monosemous (k=1): {(df['k_hdbscan']==1).sum()} / {len(df)}")

/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


In [19]:
print("=" * 60)
print("Step 4: Word Sense Induction (WSI)")
print("=" * 60)

run_wsi_for_language("english")

# Quick comparison preview
en = pd.read_csv(DATA_DIR / "english_wsi_results.csv")

print("\n── Quick Comparison Preview ──")
print(f"  EN | mean senses/lemma : {en['k_hdbscan'].mean():.3f}")
print(f"  EN | % polysemous lemmas       : {(en['k_hdbscan'] > 1).mean()*100:.1f}%")

print("\n✓ Step 4 complete.\n")

Step 4: Word Sense Induction (WSI)

  [ENGLISH] Loading embeddings…
  [ENGLISH] Running WSI for 636 lemmas…
    … 636/636 lemmas
  [ENGLISH] Saved: english_wsi_results.csv
  [ENGLISH] Saved: english_sense_labels.csv
  [ENGLISH] Mean k (HDBSCAN): 1.89  Median: 2  Monosemous (k=1): 261 / 636

── Quick Comparison Preview ──
  EN | mean senses/lemma : 1.887
  EN | % polysemous lemmas       : 59.0%

✓ Step 4 complete.



In [20]:
print("=" * 60)
print("Step 4: Word Sense Induction (WSI)")
print("=" * 60)

run_wsi_for_language("chinese")

# Quick comparison preview
zh = pd.read_csv(DATA_DIR / "chinese_wsi_results.csv")

print("\n── Quick Comparison Preview ──")
print(f"  ZH | mean senses/lemma : {zh['k_hdbscan'].mean():.3f}")
print(f"  ZH | % polysemous lemmas       : {(zh['k_hdbscan'] > 1).mean()*100:.1f}%")

print("\n✓ Step 4 complete.\n")

Step 4: Word Sense Induction (WSI)

  [CHINESE] Loading embeddings…
  [CHINESE] Running WSI for 496 lemmas…
    … 496/496 lemmas
  [CHINESE] Saved: chinese_wsi_results.csv
  [CHINESE] Saved: chinese_sense_labels.csv
  [CHINESE] Mean k (HDBSCAN): 1.23  Median: 1  Monosemous (k=1): 401 / 496

── Quick Comparison Preview ──
  ZH | mean senses/lemma : 1.230
  ZH | % polysemous lemmas       : 19.2%

✓ Step 4 complete.



# Step 5: Validation and Statistical Analysis

In [21]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from typing import Optional, Tuple

# ─── Configuration ────────────────────────────────────────────────────────────

DATA_DIR    = Path("/content") / "bible_data"
OUTPUT_DIR = Path("/content") / "output"
FIG_DIR    = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "font.family":  "DejaVu Sans",
    "font.size":    11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
})


# ─── WordNet Validation (English) ────────────────────────────────────────────

def get_wordnet_sense_counts(lemmas: list) -> dict:
    """
    Look up the number of synsets for each lemma in Princeton WordNet.
    Noun synsets only (pos='n').
    """
    try:
        from nltk.corpus import wordnet as wn
    except ImportError:
        raise ImportError("Run: pip install nltk --break-system-packages && "
                          "python -c \"import nltk; nltk.download('wordnet')\"")

    counts = {}
    for lemma in lemmas:
        synsets = wn.synsets(lemma.lower(), pos=wn.NOUN)
        counts[lemma] = len(synsets)
    return counts


def validate_english(en_results: pd.DataFrame) -> pd.DataFrame:
    """
    Correlate WSI-induced k_ward with WordNet sense counts for English nouns.
    Returns a merged DataFrame with both counts.
    """
    print("  [validate] Looking up Princeton WordNet sense counts…")
    lemmas = en_results["lemma"].tolist()
    wn_counts = get_wordnet_sense_counts(lemmas)

    en_results = en_results.copy()
    en_results["wn_senses"] = en_results["lemma"].map(wn_counts).fillna(0).astype(int)

    # Keep only lemmas with at least 1 WordNet entry
    valid = en_results[en_results["wn_senses"] > 0].copy()

    rho, p = stats.spearmanr(valid["k_hdbscan"], valid["wn_senses"])
    print(f"  [validate] EN Spearman ρ(k_hdbscan, WN_senses) = {rho:.3f}  p = {p:.4f}  "
          f"(n={len(valid)})")

    return valid, rho, p


# ─── WordNet Validation (Chinese) ────────────────────────────────────────────

def get_chinese_wordnet_sense_counts(lemmas: list) -> dict:
    """
    Look up the number of synsets for each lemma in Chinese WordNet (cmn).
    Noun synsets only (pos='n').
    """
    try:
        from nltk.corpus import wordnet as wn
    except ImportError:
        raise ImportError("Run: pip install nltk --break-system-packages && "
                          "python -c \"import nltk; nltk.download('wordnet')\"")

    counts = {}
    for lemma in lemmas:
        # 'cmn' is the language code for Mandarin Chinese in WordNet
        # Note: Chinese WordNet coverage in NLTK might be limited compared to English
        synsets = wn.synsets(lemma, lang="cmn", pos=wn.NOUN)
        counts[lemma] = len(synsets)
    return counts


def validate_chinese(zh_results: pd.DataFrame) -> pd.DataFrame:
    """
    Correlate WSI-induced k_ward with Chinese WordNet sense counts for Chinese nouns.
    Returns a merged DataFrame with both counts.
    """
    print("  [validate] Looking up Chinese WordNet sense counts…")
    lemmas = zh_results["lemma"].tolist()
    cwn_counts = get_chinese_wordnet_sense_counts(lemmas)

    zh_results = zh_results.copy()
    zh_results["cwn_senses"] = zh_results["lemma"].map(cwn_counts).fillna(0).astype(int)

    # Keep only lemmas with at least 1 WordNet entry
    valid = zh_results[zh_results["cwn_senses"] > 0].copy()

    if len(valid) == 0:
        print("  [validate] No Chinese lemmas found in NLTK's Chinese WordNet. Skipping Spearman correlation.")
        rho, p = np.nan, np.nan # No correlation if no data
    elif len(valid) == 1:
        print("  [validate] Only one Chinese lemma found in NLTK's Chinese WordNet. Skipping Spearman correlation.")
        rho, p = np.nan, np.nan # Spearman requires at least two data points
    else:
        rho, p = stats.spearmanr(valid["k_hdbscan"], valid["cwn_senses"])
        print(f"  [validate] ZH Spearman ρ(k_hdbscan, CWN_senses) = {rho:.3f}  p = {p:.4f}  "
              f"(n={len(valid)})")

    return valid, rho, p


# ─── Statistical Comparison ──────────────────────────────────────────────────

def mann_whitney_comparison(
    en_k: np.ndarray,
    zh_k: np.ndarray,
) -> dict:
    """
    Mann-Whitney U test comparing mean sense counts between EN and ZH.
    Also computes Cohen's d and the common language effect size (CLES).
    """
    u_stat, p_val = stats.mannwhitneyu(en_k, zh_k, alternative="two-sided")
    n1, n2        = len(en_k), len(zh_k)
    cles          = u_stat / (n1 * n2)  # Common Language Effect Size

    # Cohen's d (for reference alongside CLES)
    pooled_std = np.sqrt(
        ((n1 - 1) * en_k.std(ddof=1) ** 2 + (n2 - 1) * zh_k.std(ddof=1) ** 2)
        / (n1 + n2 - 2)
    )
    cohens_d = (en_k.mean() - zh_k.mean()) / (pooled_std + 1e-9)

    return {
        "en_mean_k":   round(en_k.mean(), 4),
        "zh_mean_k":   round(zh_k.mean(), 4),
        "en_median_k": round(float(np.median(en_k)), 4),
        "zh_median_k": round(float(np.median(zh_k)), 4),
        "en_std_k":    round(en_k.std(ddof=1), 4),
        "zh_std_k":    round(zh_k.std(ddof=1), 4),
        "U_statistic": round(u_stat, 2),
        "p_value":     round(p_val, 6),
        "CLES":        round(cles, 4),
        "cohens_d":    round(cohens_d, 4),
        "n_en":        int(n1),
        "n_zh":        int(n2),
    }


# ─── Figures ─────────────────────────────────────────────────────────────────

def plot_sense_distribution(df: pd.DataFrame, lang: str, col: str = "k_hdbscan") -> None:
    fig, ax = plt.subplots(figsize=(8, 4))
    counts = df[col].value_counts().sort_index()
    ax.bar(counts.index, counts.values, color="#4C72B0", edgecolor="white", linewidth=0.5)
    ax.set_xlabel("Number of Induced Senses (k)")
    ax.set_ylabel("Number of Lemmas")
    ax.set_title(f"{lang.capitalize()} — Distribution of Induced Senses per Noun Lemma")
    ax.set_xticks(range(1, df[col].max() + 1))
    fig.tight_layout()
    path = FIG_DIR / f"sense_distribution_{lang}.png"
    fig.savefig(path, dpi=150)
    plt.close(fig)
    print(f"  [fig] Saved: {path.name}")


def plot_comparison_boxplot(en_df: pd.DataFrame, zh_df: pd.DataFrame) -> None:
    combined = pd.concat([
        en_df[["k_hdbscan"]].assign(Language="English"),
        zh_df[["k_hdbscan"]].assign(Language="Chinese"),
    ])
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.violinplot(data=combined, x="Language", y="k_hdbscan",
                   palette=["#4C72B0", "#DD8452"], inner="box",
                   cut=0,   # ← add this to fix sns artifact of extending beyond 8
                   ax=ax)
    ax.set_ylabel("Induced Senses per Lemma (k, Ward)")
    ax.set_title("Distribution of Polysemy Degree: English vs. Chinese Common Nouns\n"
                 "(Bible Parallel Corpus, WSI via XLM-R + Agglomerative Clustering)")
    fig.tight_layout()
    path = FIG_DIR / "comparison_violinplot.png"
    fig.savefig(path, dpi=150)
    plt.close(fig)
    print(f"  [fig] Saved: {path.name}")


def plot_wordnet_correlation(valid_df: pd.DataFrame, rho: float, p: float) -> None:
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(valid_df["wn_senses"], valid_df["k_hdbscan"],
               alpha=0.4, s=20, color="#4C72B0")
    ax.set_xlabel("WordNet Noun Synset Count")
    ax.set_ylabel("WSI-Induced k (Ward)")
    ax.set_title(f"Validation: WSI k vs. WordNet Senses (English Nouns)\n"
                 f"Spearman ρ = {rho:.3f}, p = {p:.4f}")
    # Trend line
    m, b = np.polyfit(valid_df["wn_senses"], valid_df["k_hdbscan"], 1)
    x_line = np.linspace(valid_df["wn_senses"].min(), valid_df["wn_senses"].max(), 100)
    ax.plot(x_line, m * x_line + b, color="red", linewidth=1.5, linestyle="--")
    fig.tight_layout()
    path = FIG_DIR / "wordnet_correlation_en.png"
    fig.savefig(path, dpi=150)
    plt.close(fig)
    print(f"  [fig] Saved: {path.name}")

def plot_wordnet_correlation_chinese(valid_df: pd.DataFrame, rho: float, p: float) -> None:
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(valid_df["cwn_senses"], valid_df["k_hdbscan"],
               alpha=0.4, s=20, color="#DD8452") # Using a different color for Chinese
    ax.set_xlabel("Chinese WordNet Noun Synset Count")
    ax.set_ylabel("WSI-Induced k (Ward)")
    ax.set_title(f"Validation: WSI k vs. Chinese WordNet Senses\n"
                 f"Spearman ρ = {rho:.3f}, p = {p:.4f}")
    # Trend line
    # Only plot trend line if there's enough data and correlation is valid
    if not np.isnan(rho) and len(valid_df) > 1:
        m, b = np.polyfit(valid_df["cwn_senses"], valid_df["k_hdbscan"], 1)
        x_line = np.linspace(valid_df["cwn_senses"].min(), valid_df["cwn_senses"].max(), 100)
        ax.plot(x_line, m * x_line + b, color="red", linewidth=1.5, linestyle="--")

    fig.tight_layout()
    path = FIG_DIR / "wordnet_correlation_zh.png"
    fig.savefig(path, dpi=150)
    plt.close(fig)
    print(f"  [fig] Saved: {path.name}")


In [27]:
print("=" * 60)
print("Step 5: Validation and Statistical Analysis")
print("=" * 60)

en = pd.read_csv(DATA_DIR / "english_wsi_results.csv")
zh = pd.read_csv(DATA_DIR / "chinese_wsi_results.csv")

# ── Validation (English vs. WordNet) ──────────────────────────
valid_en, rho_en, p_en = validate_english(en)
valid_en.to_csv(OUTPUT_DIR / "validation_correlation_en.csv", index=False)

# ── Validation (Chinese vs. WordNet) ──────────────────────────
valid_zh, rho_zh, p_zh = validate_chinese(zh)
valid_zh.to_csv(OUTPUT_DIR / "validation_correlation_zh.csv", index=False)

# ── Statistical Comparison ────────────────────────────────────
stats_result = mann_whitney_comparison(
    en["k_hdbscan"].values,
    zh["k_hdbscan"].values,
)
stats_df = pd.DataFrame([stats_result])
stats_df.to_csv(OUTPUT_DIR / "statistical_comparison.csv", index=False)

print("\n── Statistical Comparison Results ──")
for k, v in stats_result.items():
    print(f"  {k:22s}: {v}")

# ── Figures ───────────────────────────────────────────────────
plot_sense_distribution(en, "english")
plot_sense_distribution(zh, "chinese")
plot_comparison_boxplot(en, zh)
plot_wordnet_correlation(valid_en, rho_en, p_en)
# You might want to create a plot for Chinese WordNet correlation as well, if coverage is good
plot_wordnet_correlation_chinese(valid_zh, rho_zh, p_zh) # This would require a new function

# ── Interpretation ────────────────────────────────────────────
print("\n── Interpretation ──")
if stats_result["p_value"] < 0.05:
    direction = "English" if stats_result["en_mean_k"] > stats_result["zh_mean_k"] else "Chinese"
    print(f"  Significant difference found (p={stats_result['p_value']:.4f}).")
    print(f"  {direction} nouns show higher mean polysemy degree.")
else:
    print(f"  No significant difference found (p={stats_result['p_value']:.4f}).")

d = abs(stats_result["cohens_d"])
magnitude = "small" if d < 0.2 else ("medium" if d < 0.5 else "large")
print(f"  Effect size: Cohen's d = {stats_result['cohens_d']:.3f} ({magnitude})")
print(f"  Spearman ρ (EN WSI vs. WordNet): {rho_en:.3f} (p={p_en:.4f})")
print(f"  Spearman ρ (ZH WSI vs. WordNet): {rho_zh:.3f} (p={p_zh:.4f})")

print("\n✓ Step 5 complete.\n")

Step 5: Validation and Statistical Analysis
  [validate] Looking up Princeton WordNet sense counts…
  [validate] EN Spearman ρ(k_hdbscan, WN_senses) = 0.143  p = 0.0003  (n=630)
  [validate] Looking up Chinese WordNet sense counts…


/usr/local/lib/python3.12/dist-packages/nltk/corpus/reader/wordnet.py:2214: UserWarning: cmn: invalid offset 14869976-n in '14869976-n	cmn:lemma	污点
'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/nltk/corpus/reader/wordnet.py:2214: UserWarning: cmn: invalid offset 14869977-n in '14869977-n	cmn:lemma	小斑
'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/nltk/corpus/reader/wordnet.py:2214: UserWarning: cmn: invalid offset 15168570-n in '15168570-n	cmn:lemma	规定的睡觉时间
'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/nltk/corpus/reader/wordnet.py:2214: UserWarning: cmn: invalid offset 15171146-n in '15171146-n	cmn:lemma	节日
'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/nltk/corpus/reader/wordnet.py:2214: UserWarning: cmn: invalid offset 15171147-n in '15171147-n	cmn:lemma	纪念日
'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/nltk/corpus/reader/wordnet.py:2214: UserWarning: cmn: invalid offset 15171739-n in '15171739-n	cmn:lemma	竞技状态不佳的日子
'
  

  [validate] ZH Spearman ρ(k_hdbscan, CWN_senses) = 0.072  p = 0.4091  (n=134)

── Statistical Comparison Results ──
  en_mean_k             : 1.8868
  zh_mean_k             : 1.2298
  en_median_k           : 2.0
  zh_median_k           : 1.0
  en_std_k              : 1.075
  zh_std_k              : 0.5427
  U_statistic           : 223322.5
  p_value               : 0.0
  CLES                  : 0.7079
  cohens_d              : 0.7446
  n_en                  : 636
  n_zh                  : 496
  [fig] Saved: sense_distribution_english.png
  [fig] Saved: sense_distribution_chinese.png
  [fig] Saved: comparison_violinplot.png
  [fig] Saved: wordnet_correlation_en.png
  [fig] Saved: wordnet_correlation_zh.png

── Interpretation ──
  Significant difference found (p=0.0000).
  English nouns show higher mean polysemy degree.
  Effect size: Cohen's d = 0.745 (large)
  Spearman ρ (EN WSI vs. WordNet): 0.143 (p=0.0003)
  Spearman ρ (ZH WSI vs. WordNet): 0.072 (p=0.4091)

✓ Step 5 complete.



## 5.1 WordNet Coverage

In [ ]:
import nltk
from nltk.corpus import wordnet as wn
import pandas as pd
from pathlib import Path

# Ensure the 'en' DataFrame is loaded, if not already in memory
# (assuming DATA_DIR is defined from previous cells)
if 'en' not in locals():
    DATA_DIR = Path("/content") / "bible_data"
    en = pd.read_csv(DATA_DIR / "english_wsi_results.csv")

english_lemmas = en['lemma'].tolist()
n_with_synsets_en = sum(1 for lemma in english_lemmas
                          if len(wn.synsets(lemma.lower(), pos=wn.NOUN)) > 0)

total_lemmas_en = len(english_lemmas)
print(f"English WordNet Coverage: {n_with_synsets_en}/{total_lemmas_en} = {n_with_synsets_en/total_lemmas_en:.1%}")


English WordNet Coverage: 630/636 = 99.1%


In [ ]:
import nltk
from nltk.corpus import wordnet as wn
import pandas as pd

# Ensure the 'zh' DataFrame is loaded, if not already in memory
# (assuming DATA_DIR is defined from previous cells)
if 'zh' not in locals():
    DATA_DIR = Path("/content") / "bible_data"
    zh = pd.read_csv(DATA_DIR / "chinese_wsi_results.csv")

chinese_lemmas = zh['lemma'].tolist()
n_with_synsets = sum(1 for lemma in chinese_lemmas
                     if len(wn.synsets(lemma, lang="cmn")) > 0)

total_lemmas = len(chinese_lemmas)
print(f"Chinese WordNet Coverage: {n_with_synsets}/{total_lemmas} = {n_with_synsets/total_lemmas:.1%}")

Chinese WordNet Coverage: 142/496 = 28.6%


# Step 6: Qualitative Analysis

In [28]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict

# ─── Configuration ────────────────────────────────────────────────────────────

# DATA_DIR and OUTPUT_DIR are defined in previous cells and are globally accessible.

TOP_N_WORDS       = 20   # Top N most polysemous lemmas per language
EXAMPLES_PER_SENSE = 2   # Number of example verses per cluster

# ─── Sense Example Retrieval (Corrected) ─────────────────────────────────────────────────

def get_sense_examples(
    lang: str,
    top_n: int = TOP_N_WORDS,
    examples_per_sense: int = EXAMPLES_PER_SENSE,
) -> str:
    """
    For the top_n most polysemous lemmas, retrieve example contexts
    for each induced sense cluster.

    Returns a formatted string ready for a paper's qualitative appendix.
    """
    results_df = pd.read_csv(DATA_DIR / f"{lang}_wsi_results.csv")
    labels_df  = pd.read_csv(DATA_DIR / f"{lang}_sense_labels.csv")
    nouns_df   = pd.read_csv(DATA_DIR / f"{lang}_nouns.csv")

    # Top polysemous by k_hdbscan (was k_ward)
    poly = results_df[results_df["k_hdbscan"] > 1].nlargest(top_n, "k_hdbscan")

    # Merge labels with original contexts
    merged = labels_df.merge(
        nouns_df[["lemma", "verse_id", "context"]].drop_duplicates(),
        on=["lemma", "verse_id"],
        how="left",
    )

    lines = []
    lines.append(f"{'='*60}")
    lines.append(f"QUALITATIVE SENSE ANALYSIS — {lang.upper()}")
    # Updated title to reflect HDBSCAN
    lines.append(f"Top {top_n} Most Polysemous Nouns (WSI, HDBSCAN Clustering)")
    lines.append(f"{'='*60}\n")

    for _, row in poly.iterrows():
        lemma = row["lemma"]
        # Use k_hdbscan
        k     = int(row["k_hdbscan"])
        n_occ = int(row["n_instances"])
        # silhouette_ward is not available for HDBSCAN, so remove or set to N/A
        # sil   = row.get("silhouette_ward", "N/A") # Removed

        # Updated output line, removed silhouette score
        lines.append(f"Lemma: '{lemma}'  |  k={k}  |  n={n_occ}")
        lines.append("-" * 50)

        lemma_data = merged[merged["lemma"] == lemma]

        for cluster_id in range(k):
            # Use cluster_hdbscan
            cluster_rows = lemma_data[lemma_data["cluster_hdbscan"] == cluster_id]
            lines.append(f"  Sense {cluster_id + 1} ({len(cluster_rows)} occurrences):")

            # Sample diverse examples
            sample = cluster_rows.dropna(subset=["context"]).head(examples_per_sense)
            for _, ex in sample.iterrows():
                ctx = str(ex["context"])[:200].replace("\n", " ")
                lines.append(f"    • {ex['verse_id']}: {ctx}")

        lines.append("")

    return "\n".join(lines)


# ─── LaTeX Table Generation (Corrected) ──────────────────────────────────────────────────

def generate_latex_table(top_n: int = 15) -> str:
    """
    Generate a LaTeX longtable comparing top polysemous nouns in both languages.
    Format:
      Rank | English Lemma | EN k | Chinese Lemma | ZH k
    """
    en = pd.read_csv(DATA_DIR / "english_wsi_results.csv")
    zh = pd.read_csv(DATA_DIR / "chinese_wsi_results.csv")

    # Use k_hdbscan for nlargest and column selection
    en_top = en.nlargest(top_n, "k_hdbscan")[["lemma", "k_hdbscan", "n_instances"]].reset_index(drop=True)
    zh_top = zh.nlargest(top_n, "k_hdbscan")[["lemma", "k_hdbscan", "n_instances"]].reset_index(drop=True)

    lines = [
        r"\begin{table}[h]",
        r"\centering",
        r"\caption{Top Polysemous Common Nouns by Induced Sense Count (k, HDBSCAN): English vs. Chinese}", # Updated caption
        r"\label{tab:top_polysemous}",
        r"\begin{tabular}{clccclcc}",
        r"\toprule",
        r"Rank & English Lemma & EN $k$ & EN $n$ & & Chinese Lemma & ZH $k$ & ZH $n$ \\",
        r"\midrule",
    ]

    for i in range(top_n):
        en_row = en_top.iloc[i] if i < len(en_top) else None
        zh_row = zh_top.iloc[i] if i < len(zh_top) else None

        en_lemma = en_row["lemma"]                  if en_row is not None else ""
        en_k     = int(en_row["k_hdbscan"])            if en_row is not None else "" # Use k_hdbscan
        en_n     = int(en_row["n_instances"])     if en_row is not None else ""
        zh_lemma = zh_row["lemma"]                  if zh_row is not None else ""
        zh_k     = int(zh_row["k_hdbscan"])            if zh_row is not None else "" # Use k_hdbscan
        zh_n     = int(zh_row["n_instances"])     if zh_row is not None else ""

        lines.append(
            f"{i+1} & {en_lemma} & {en_k} & {en_n} & & {zh_lemma} & {zh_k} & {zh_n} \\"
        )

    lines += [
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table}",
    ]
    return "\n".join(lines)

# ─── Wide Comparison Profile (Copied for context) ─────────────────────────────────────────────────

def generate_comparison_profile() -> pd.DataFrame:
    """
    Create a summary comparison table for paper Table 2.
    """
    en = pd.read_csv(DATA_DIR / "english_wsi_results.csv")
    zh = pd.read_csv(DATA_DIR / "chinese_wsi_results.csv")

    def profile(df: pd.DataFrame, lang: str) -> dict:
        return {
            "Language":            lang,
            "Total lemmas":        len(df),
            "Mean k (HDBSCAN)":    round(df["k_hdbscan"].mean(), 3), # Use k_hdbscan
            "Median k (HDBSCAN)":  round(df["k_hdbscan"].median(), 3), # Use k_hdbscan
            "Std k (HDBSCAN)":     round(df["k_hdbscan"].std(ddof=1), 3), # Use k_hdbscan
            "% Monosemous (k=1)":  round((df["k_hdbscan"] == 1).mean() * 100, 1), # Use k_hdbscan
            "% Polysemous (k>1)":  round((df["k_hdbscan"] > 1).mean() * 100, 1), # Use k_hdbscan
            "Max k":               int(df["k_hdbscan"].max()), # Use k_hdbscan
        }

    rows = [profile(en, "English"), profile(zh, "Chinese")]
    return pd.DataFrame(rows)


print("=" * 60)
print("Step 6: Qualitative Analysis")
print("=" * 60)

# ── Sense examples ────────────────────────────────────────────
for lang in ["english", "chinese"]:
    text = get_sense_examples(lang)
    out  = OUTPUT_DIR / f"qualitative_{lang}_top_polysemous.txt"
    out.write_text(text, encoding="utf-8")
    print(f"  [saved] {out.name}")

# ── LaTeX table ───────────────────────────────────────────────
latex = generate_latex_table(top_n=15)
tex_path = OUTPUT_DIR / "table_top_polysemous_latex.tex"
tex_path.write_text(latex, encoding="utf-8")
print(f"  [saved] {tex_path.name}")

# ── Comparison profile ────────────────────────────────────────
profile = generate_comparison_profile()
csv_path = OUTPUT_DIR / "polysemy_profile_comparison.csv"
profile.to_csv(csv_path, index=False)
print(f"  [saved] {csv_path.name}")
print()
print(profile.to_string(index=False))

print("\n✓ Step 6 complete.\n")

Step 6: Qualitative Analysis
  [saved] qualitative_english_top_polysemous.txt
  [saved] qualitative_chinese_top_polysemous.txt
  [saved] table_top_polysemous_latex.tex
  [saved] polysemy_profile_comparison.csv

Language  Total lemmas  Mean k (HDBSCAN)  Median k (HDBSCAN)  Std k (HDBSCAN)  % Monosemous (k=1)  % Polysemous (k>1)  Max k
 English           636             1.887                 2.0            1.075                41.0                59.0      9
 Chinese           496             1.230                 1.0            0.543                80.8                19.2      5

✓ Step 6 complete.



# Download data

In [ ]:
!zip -r output_niv_cuv.zip /content/output/

In [ ]:
!zip -r data_niv_cuv.zip /content/bible_data/